In [10]:
"""Titanic survival analysis - answers Q1 to Q4.

Usage: python3 titanic_analysis.py [path/to/Titanic-Dataset.csv]
Figures are written to ./outputs/
"""
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

CSV = sys.argv[1] if len(sys.argv) > 1 and not sys.argv[1].startswith('-f') else "/content/sample_data/Titanic-Dataset (1).csv"

# Fallback for notebook environments where __file__ is not defined
try:
    _file_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _file_dir = os.path.abspath(".")

OUT = os.path.join(_file_dir, "outputs")
os.makedirs(OUT, exist_ok=True)

# ---- style ---------------------------------------------------------------
SURFACE, INK, INK2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, BASE = "#e1e0d9", "#c3c2b7"
C_MALE, C_FEMALE = "#2a78d6", "#eb6834"  # categorical slots 1 and 2
COL = {"male": C_MALE, "female": C_FEMALE}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "sans-serif", "font.size": 10,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": BASE, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
})


def style(ax, grid_axis="y"):
    ax.grid(False)
    ax.grid(True, axis=grid_axis)
    ax.tick_params(length=0)


df = pd.read_csv(CSV)
df["Sex"] = df["Sex"].str.lower()
df["Class"] = df["Pclass"].map({1: "1st", 2: "2nd", 3: "3rd"})

# ===========================================================================
# Q1  Survivors vs age, split by gender
# ===========================================================================
bins = np.arange(0, 90, 10)
labels = [f"{b}-{b+9}" for b in bins[:-1]]
known = df.dropna(subset=["Age"]).copy()
known["AgeBin"] = pd.cut(known["Age"], bins=bins, labels=labels, right=False)

surv = known[known["Survived"] == 1]
counts = surv.groupby(["AgeBin", "Sex"], observed=False).size().unstack(fill_value=0)[["male", "female"]]
rate = known.groupby(["AgeBin", "Sex"], observed=False)["Survived"].mean().unstack()[["male", "female"]]
n_bin = known.groupby(["AgeBin", "Sex"], observed=False).size().unstack()[["male", "female"]]

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(labels))
w = 0.38
for i, sx in enumerate(["male", "female"]):
    bars = a.bar(x + (i - 0.5) * w, counts[sx], w - 0.04, color=COL[sx], label=sx.capitalize())
    for r, v in zip(bars, counts[sx]):
        if v > 0:
            a.text(r.get_x() + r.get_width() / 2, v + 0.8, str(v), ha="center", va="bottom",
                   fontsize=8, color=INK2)
a.set_xticks(x, labels)
a.set_xlabel("Age group (years)")
a.set_ylabel("Number of survivors")
a.set_title("Survivors by age group and gender")
a.legend(frameon=False, loc="upper right")
style(a)

for sx in ["male", "female"]:
    ok = n_bin[sx] >= 5  # hide very small groups
    b.plot(x[ok], rate[sx][ok] * 100, color=COL[sx], lw=2, marker="o", ms=7,
           markeredgecolor=SURFACE, markeredgewidth=2, label=sx.capitalize())
b.set_xticks(x, labels)
b.set_ylim(0, 100)
b.set_xlabel("Age group (years)")
b.set_ylabel("Survival rate (%)")
b.set_title("Survival rate by age group and gender")
b.legend(frameon=False, loc="upper right")
style(b)
fig.suptitle("Q1  Survival vs age, by gender", x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.text(0.01, 0.005, f"{len(known)} of {len(df)} passengers have a recorded age; the rest are excluded. "
         "Age groups with fewer than 5 passengers are not drawn as rate points.",
         fontsize=8, color=MUTED)
fig.tight_layout(rect=(0, 0.03, 1, 0.95))
fig.savefig(os.path.join(OUT, "q1_survival_vs_age_by_gender.png"), dpi=160)
plt.show()

# ===========================================================================
# Q2  Fare vs passenger class, with gender (colour) and age (size)
# ===========================================================================
q2 = df.dropna(subset=["Age"]).copy()
q2 = q2[q2["Fare"] > 0]  # log scale cannot show 0
rng = np.random.default_rng(7)
q2["xj"] = q2["Pclass"] + rng.uniform(-0.28, 0.28, len(q2))

fig, ax = plt.subplots(figsize=(10, 6.5))
for sx in ["male", "female"]:
    d = q2[q2["Sex"] == sx]
    ax.scatter(d["xj"], d["Fare"], s=8 + d["Age"] * 1.6, color=COL[sx], alpha=0.55,
               edgecolor=SURFACE, linewidth=0.6, label=sx.capitalize())
for p in (1, 2, 3):
    for sx in ["male", "female"]:
        med = q2[(q2["Pclass"] == p) & (q2["Sex"] == sx)]["Fare"].median()
        x0, x1 = (p - 0.30, p - 0.02) if sx == "male" else (p + 0.02, p + 0.30)
        ax.hlines(med, x0, x1, color=INK, lw=2.2, zorder=5)
        ax.text((x0 + x1) / 2, med * 1.07, f"${med:.0f}", ha="center", va="bottom", fontsize=9,
                fontweight="bold", color=INK, zorder=6,
                bbox=dict(boxstyle="round,pad=0.12", fc=SURFACE, ec="none", alpha=0.85))
ax.set_yscale("log")
ax.set_yticks([5, 10, 20, 50, 100, 200, 500], ["$5", "$10", "$20", "$50", "$100", "$200", "$500"])
ax.set_xticks([1, 2, 3], ["1st class", "2nd class", "3rd class"])
ax.set_xlim(0.45, 3.55)
ax.set_ylabel("Fare (log scale)")
ax.set_xlabel("Passenger class")
ax.set_title("Q2  Fare by passenger class, with gender and age", loc="left", fontsize=13)
style(ax)
ax.grid(True, axis="y", which="major")

handles = [Line2D([], [], marker="o", ls="", color=COL["male"], label="Male", markersize=8),
           Line2D([], [], marker="o", ls="", color=COL["female"], label="Female", markersize=8)]
for age in (5, 30, 60):
    handles.append(Line2D([], [], marker="o", ls="", color=MUTED, alpha=0.7,
                          markersize=np.sqrt(8 + age * 1.6), label=f"Age {age}"))
ax.legend(handles=handles, frameon=False, loc="upper right", ncol=1, title="Colour = gender, size = age",
          title_fontsize=8, fontsize=9)
fig.text(0.01, 0.005, f"{len(q2)} passengers shown (known age, fare above $0). Points are jittered sideways "
         "within each class. Black bars with labels = median fare (male left, female right).",
         fontsize=8, color=MUTED)
fig.tight_layout(rect=(0, 0.025, 1, 1))
fig.savefig(os.path.join(OUT, "q2_fare_vs_class_gender_age.png"), dpi=160)
plt.show()

# ===========================================================================
# Q3  Survival rate: sex x class
# ===========================================================================
sr = df.groupby(["Class", "Sex"])["Survived"].mean().unstack()[["male", "female"]] * 100
n = df.groupby(["Class", "Sex"]).size().unstack()[["male", "female"]]

fig, ax = plt.subplots(figsize=(8.5, 5.5))
x = np.arange(3)
for i, sx in enumerate(["male", "female"]):
    bars = ax.bar(x + (i - 0.5) * 0.36, sr[sx].values, 0.32, color=COL[sx], label=sx.capitalize())
    for r, v, n_val in zip(bars, sr[sx].values, n[sx].values):
        ax.text(r.get_x() + r.get_width() / 2, v + 1.2, f"{v:.0f}%", ha="center", va="bottom",
                fontsize=10, color=INK, fontweight="bold")
        ax.text(r.get_x() + r.get_width() / 2, 2, f"n={n_val}", ha="center", va="bottom", fontsize=8, color="#ffffff")
ax.set_xticks(x, ["1st class", "2nd class", "3rd class"])
ax.set_ylim(0, 105)
ax.set_ylabel("Survival rate (%)")
ax.set_xlabel("Passenger class")
ax.set_title("Q3  Survival rate of men and women by passenger class", loc="left", fontsize=13)
ax.legend(frameon=False, loc="upper right")
style(ax)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "q3_survival_rate_by_sex_and_class.png"), dpi=160)
plt.show()

# ===========================================================================
# Q4  Who was more likely to survive?
# ===========================================================================
d = df.copy()
d["Title"] = d["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
d["Title"] = d["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
d["Title"] = d["Title"].where(d["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Other")
d["Family"] = d["SibSp"] + d["Parch"] + 1
d["FamGroup"] = pd.cut(d["Family"], [0, 1, 4, 20], labels=["Alone", "2-4", "5+"])
d["AgeGroup"] = pd.cut(d["Age"], [0, 12, 18, 35, 60, 100],
                       labels=["Child (<13)", "Teen (13-18)", "Adult (19-35)", "Adult (36-60)", "Senior (60+)"])
d["FareBand"] = pd.qcut(d["Fare"], 4, labels=["Q1 (cheapest)", "Q2", "Q3", "Q4 (priciest)"])


def rate_by(col, order=None):
    g = d.groupby(col, observed=True)["Survived"].agg(["mean", "size"])
    if order is not None:
        g = g.loc[order]
    return g


panels = [
    ("Gender", rate_by("Sex", ["female", "male"]), ["Female", "Male"]),
    ("Passenger class", rate_by("Class"), None),
    ("Age group", rate_by("AgeGroup"), None),
    ("Title in name", rate_by("Title", ["Mrs", "Miss", "Master", "Other", "Mr"]), None),
    ("Travelling with family", rate_by("FamGroup"), None),
    ("Fare quartile", rate_by("FareBand"), None),
]
overall = d["Survived"].mean() * 100

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (title, g, lab) in zip(axes.flat, panels):
    vals = g["mean"].values * 100
    names = lab or [str(i) for i in g.index]
    cols = [C_FEMALE if n_name in ("Female",) else C_MALE if n_name == "Male" else "#2a78d6" for n_name in names]
    bars = ax.bar(range(len(vals)), vals, 0.6, color=cols)
    for r, v, n_val in zip(bars, vals, g["size"].values):
        ax.text(r.get_x() + r.get_width() / 2, v + 1.5, f"{v:.0f}%", ha="center", va="bottom",
                fontsize=9, fontweight="bold")
        ax.text(r.get_x() + r.get_width() / 2, -9, f"n={n_val}", ha="center", va="top", fontsize=7, color=MUTED)
    ax.axhline(overall, color=MUTED, lw=1, ls="--")
    ax.set_xticks(range(len(vals)), names, fontsize=8)
    ax.set_ylim(0, 100)
    ax.set_title(title)
    style(ax)
for ax in axes[:, 0]:
    ax.set_ylabel("Survival rate (%)")
fig.suptitle("Q4  Who was more likely to survive?", x=0.01, ha="left", fontsize=14, fontweight="bold")
fig.text(0.01, 0.005, f"Dashed line = overall survival rate ({overall:.0f}%). n = passengers in each group; "
         "age panel excludes passengers with no recorded age.", fontsize=8, color=MUTED)
fig.tight_layout(rect=(0, 0.02, 1, 0.96))
fig.savefig(os.path.join(OUT, "q4_survival_factors.png"), dpi=160)
plt.show()

# ---- printed summary for the write-up -------------------------------------
pd.set_option("display.width", 140)
print(f"Overall survival: {overall:.1f}%  ({int(d['Survived'].sum())} of {len(d)})\n")
for title, g, _ in panels:
    print(title)
    print((g.assign(rate=(g['mean'] * 100).round(1))[["rate", "size"]]).to_string(), "\n")
print("Sex x Class survival % (n)")
print(sr.round(1).to_string(), "\n")
print(n.to_string(), "\n")
print("Median age survivors vs not:", d.groupby("Survived")["Age"].median().to_dict())
print("Child (<13) by class:")
print(d[d["Age"] < 13].groupby("Class")["Survived"].agg(["mean", "size"]).round(2).to_string())
print("\nQ1 survivors by age bin / sex")
print(counts.to_string())
print("\nQ2 median fare (excl 0) by class / sex")
print(q2.groupby(["Pclass", "Sex"])["Fare"].median().round(1).unstack().to_string())

Overall survival: 38.4%  (342 of 891)

Gender
        rate  size
Sex               
female  74.2   314
male    18.9   577 

Passenger class
       rate  size
Class            
1st    63.0   216
2nd    47.3   184
3rd    24.2   491 

Age group
               rate  size
AgeGroup                 
Child (<13)    58.0    69
Teen (13-18)   42.9    70
Adult (19-35)  38.3   358
Adult (36-60)  40.0   195
Senior (60+)   22.7    22 

Title in name
        rate  size
Title             
Mrs     79.4   126
Miss    70.3   185
Master  57.5    40
Other   34.8    23
Mr      15.7   517 

Travelling with family
          rate  size
FamGroup            
Alone     30.4   537
2-4       57.9   292
5+        16.1    62 

Fare quartile
               rate  size
FareBand                 
Q1 (cheapest)  19.7   223
Q2             30.4   224
Q3             45.5   222
Q4 (priciest)  58.1   222 

Sex x Class survival % (n)
Sex    male  female
Class              
1st    36.9    96.8
2nd    15.7    92.1
3rd    13.5    5

### Additional Titanic Visualizations
Let's explore some more dimensions of the data: survival rate by embarkation port, family size distribution, and a boxplot of fares by class.

In [13]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Survival Rate by Embarkation Port
emb_data = df.groupby('Embarked')['Survived'].agg(['mean', 'size']).reset_index()
emb_data['mean'] = emb_data['mean'] * 100
emb_ports = {'C': 'Cherbourg', 'Q': 'Queenstown', 'S': 'Southampton'}
emb_data['PortName'] = emb_data['Embarked'].map(emb_ports)

bars1 = axes[0].bar(emb_data['PortName'], emb_data['mean'], color='#4a90e2', width=0.5)
for r, v in zip(bars1, emb_data['mean']):
    axes[0].text(r.get_x() + r.get_width()/2, v + 2, f"{v:.1f}%", ha='center', fontweight='bold')
axes[0].set_title("Survival Rate by Embarkation Port")
axes[0].set_ylabel("Survival Rate (%)")
axes[0].set_ylim(0, 100)
style(axes[0])

# 2. Family Size Distribution (SibSp + Parch + 1)
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
family_counts = df['FamilySize'].value_counts().sort_index()
axes[1].bar(family_counts.index, family_counts.values, color='#5cbae6', width=0.6)
axes[1].set_title("Distribution of Family Sizes")
axes[1].set_xlabel("Family Members Onboard")
axes[1].set_ylabel("Passenger Count")
axes[1].set_xticks(family_counts.index)
style(axes[1])

# 3. Boxplot of Fares by Class (Excluding Zero Fares)
fare_data = df[df['Fare'] > 0]
classes = ['1st', '2nd', '3rd']
fare_by_class = [fare_data[fare_data['Class'] == c]['Fare'] for c in classes]

axes[2].boxplot(fare_by_class, tick_labels=classes, patch_artist=True,
                boxprops=dict(facecolor='#b2d8d8', color=BASE),
                medianprops=dict(color=INK, linewidth=1.5))
axes[2].set_title("Fare Distribution by Class")
axes[2].set_ylabel("Fare (Ticket Price)")
axes[2].set_yscale('log')
style(axes[2], grid_axis='y')

plt.tight_layout()
plt.show()